In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/course-video.csv
/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/user-course-comment.csv
/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/course-teacher.csv
/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/user.csv
/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/course.csv
/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/course-exercise.csv
/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/course-school.csv
/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/course_comment.csv
/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/user-video.csv
/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/course-concept.csv
/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/video.csv
/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/user-course.csv
/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/user-course-reply.csv
/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/course-fiel

In [2]:
import os
import shutil
import json
import re
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import torch
from IPython.display import FileLink

print("Đã khởi tạo xong môi trường!")

Đã khởi tạo xong môi trường!


In [3]:
course = pd.read_csv('/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/course.csv')
user = pd.read_csv('/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/user.csv')
video = pd.read_csv('/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/video.csv')

In [4]:
print(f'course: {course.shape}')
print(f'user: {user.shape}')
print(f'video: {video.shape}')

course: (3781, 4)
user: (3330294, 2)
video: (1316090, 3)


In [5]:
user_video = pd.read_csv('/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/user-video.csv')
user_course = pd.read_csv('/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/user-course.csv')
user_exercise = pd.read_csv('/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/user-exercise.csv')

In [6]:
print(f'user_video: {user_video.shape}')
print(f'user_course: {user_course.shape}')
print(f'user_exercise: {user_exercise.shape}')

user_video: (1981001, 4)
user_course: (11807090, 3)
user_exercise: (427207, 3)


In [7]:
course_school = pd.read_csv('/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/course-school.csv')
course_teacher = pd.read_csv('/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/course-teacher.csv')
course_field = pd.read_csv('/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/course-field.csv')
course_video = pd.read_csv('/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/course-video.csv')

In [8]:
print(f'course_school: {course_school.shape}')
print(f'course_teacher: {course_teacher.shape}')
print(f'course_field: {course_field.shape}')
print(f'course_video: {course_video.shape}')

course_school: (3816, 6)
course_teacher: (97173, 6)
course_field: (793, 2)
course_video: (85787, 2)


In [9]:
course_exercise = pd.read_csv('/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/course-exercise.csv')
course_concept = pd.read_csv('/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/course-concept.csv')
course_comment = pd.read_csv('/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/course_comment.csv')

In [10]:
print(f'course_exercise: {course_exercise.shape}')
print(f'course_concept: {course_concept.shape}')
print(f'course_comment: {course_comment.shape}')

course_exercise: (66704, 3)
course_concept: (451073, 4)
course_comment: (2980, 2)


In [11]:
user_course_comment = pd.read_csv('/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/user-course-comment.csv')
user_course_reply = pd.read_csv('/kaggle/input/datasets/ngocccchannn/data-mooc-nofilter/user-course-reply.csv')

In [12]:
print(f'user_course_comment: {user_course_comment.shape}')
print(f'user_course_reply: {user_course_reply}')

user_course_comment: (758672, 3)
user_course_reply:           user_id  course_id  user_course_num_replies
0        U_100306   C_697018                        1
1      U_10032046   C_766192                       26
2      U_10034442   C_696889                        1
3      U_10036301   C_936971                        1
4      U_10036304   C_936971                        1
...           ...        ...                      ...
28176   U_9959991   C_682155                        7
28177   U_9963608   C_936971                        1
28178   U_9969882   C_696700                        2
28179     U_99772  C_1903985                        1
28180     U_99772   C_697032                        1

[28181 rows x 3 columns]


In [13]:
user.head()

,user_id,gender
0,U_22,0
1,U_24,1
2,U_25,0
3,U_53,1
4,U_54,2


In [14]:
# Thay đổi tên cột cho tập user
user = user.rename(columns={
    'gender':'user_gender'
})
user

,user_id,user_gender
0,U_22,0
1,U_24,1
2,U_25,0
3,U_53,1
4,U_54,2
...,...,...
3330289,U_34712108,0
3330290,U_34712111,2
3330291,U_34712112,1
3330292,U_34712114,1


In [15]:
course.columns.tolist()

['course_id', 'name_trans', 'prerequisites_trans', 'about_trans']

In [16]:
# Thay đổi tên cột cho tập course 
course = course[['course_id','name_trans','prerequisites_trans','about_trans']]
course = course.rename(columns={
    'name_trans':'course_name',
    'prerequisites_trans':'course_prerequisites',
    'about_trans':'course_about',
})
course.head()

,course_id,course_name,course_prerequisites,course_about
0,C_584313,"introduction to ""zi zhi tong jian""",NaN,"through the teacher's guidance, students can g..."
1,C_584329,calculus - limit theory and functions of one v...,NaN,this course is a basic mathematics course in s...
2,C_584381,photojournalism,NaN,"master basic photography skills, understand ho..."
3,C_597208,data mining: theory and algorithms,NaN,the most interesting theory + the most useful ...
4,C_597225,university computer,NaN,university computer courses will be guided by ...


In [17]:
# xóa những khóa học mẫu, khóa học test (được xác định thủ công trước đó)
print(f'Kích thước gốc: {course.shape}')
test_course_id = ['C_1073350','C_1721804','C_1734933','C_1770893','C_1824311',
                  'C_608132','C_645178','C_654397','C_654435','C_681505',
                  'C_789961','C_920114']
course = course[~course['course_id'].isin(test_course_id)]
print(f'Sau khi xóa khóa học mẫu: {course.shape}')

Kích thước gốc: (3781, 4)
Sau khi xóa khóa học mẫu: (3769, 4)


In [18]:
# Chuẩn hóa giá trị về string và xử lý điều kiện
NEG = re.compile(r"^(none|no|not required|not needed|nothing|n/a)\b", re.I) 
  
def norm_prereq(x): 
    if pd.isna(x) or not str(x).strip(): 
        return "not stated"          # không rõ 
    s = str(x).strip() 
    return "none" if NEG.match(s) else s   # "none" = ghi rõ không cần tiên quyết 
  
course["course_prerequisites"] = course["course_prerequisites"].map(norm_prereq) 

In [19]:
print(f'Course: {course.shape}')
print(f'Course-Field: {course_field.shape}')
course_field

Course: (3769, 4)
Course-Field: (793, 2)


,course_id,course_field
0,C_681001,geography
1,C_680926,art
2,C_629518,history
3,C_697091,computer science and technology
4,C_680955,foreign languages ​​and literature
...,...,...
788,C_697288,crop science
789,C_629573,computer science and technology
790,C_1766899,physics
791,C_597265,foreign languages ​​and literature


In [20]:
# Xử lý các ký tự ngắt dòng '\u200b' bằng ''
course_field = course_field[['course_id','course_field']]
course_field['course_field'] = course_field['course_field'].apply(
    lambda x: str(x).replace('\u200b', '') if pd.notna(x) else x
)

In [21]:
# Gom nhóm lĩnh vực cho từng khóa học trong tập 'course_field' và gộp với tập course 
course_field = course_field.drop_duplicates().groupby('course_id')['course_field'].agg(list).reset_index()
course = pd.merge(course, course_field, on='course_id',how='left')
print(course.shape)
course

(3769, 5)


,course_id,course_name,course_prerequisites,course_about,course_field
0,C_584313,"introduction to ""zi zhi tong jian""",not stated,"through the teacher's guidance, students can g...","[chinese language and literature, history]"
1,C_584329,calculus - limit theory and functions of one v...,not stated,this course is a basic mathematics course in s...,"[applied economics, math, theoretical economic..."
2,C_584381,photojournalism,not stated,"master basic photography skills, understand ho...","[art, journalism and communication]"
3,C_597208,data mining: theory and algorithms,not stated,the most interesting theory + the most useful ...,[computer science and technology]
4,C_597225,university computer,not stated,university computer courses will be guided by ...,NaN
...,...,...,...,...,...
3764,C_2338076,(wind project) software engineering,not stated,project blast 2021 [first phase recruitment] i...,NaN
3765,C_2341259,(wind project) object-oriented programming (c++),not stated,project blast 2021 [first phase recruitment] i...,NaN
3766,C_2337996,(wind project) data structure (part 2),not stated,project blast 2021 [first phase recruitment] i...,NaN
3767,C_1945689,machine learning bootcamp,not stated,teacher zhang min from tsinghua university wil...,NaN


In [22]:
print(course_school.shape)
course_school.columns.tolist()

(3816, 6)


['course_id', 'school_id', 'name_en', 'sign', 'school_about', 'motto']

In [23]:
# Thay đổi tên cột cho tập course_school
course_school = course_school[['course_id', 'school_id', 'name_en', 'school_about', 'motto']]
course_school = course_school.rename(columns = {
    'name_en': 'school_name',
    'motto': 'school_motto'
})
print(course_school.shape)
course_school.columns.tolist()

(3816, 5)


['course_id', 'school_id', 'school_name', 'school_about', 'school_motto']

In [24]:
# Kiểm tra dữ liệu course_teacher 
print(course_teacher.shape)
print(course_teacher.columns.tolist())
course_teacher.head()

(97173, 6)
['teacher_id', 'teacher_name', 'teacher_about', 'teacher_job_title', 'teacher_org_name', 'course_id']


,teacher_id,teacher_name,teacher_about,teacher_job_title,teacher_org_name,course_id
0,T_1,Liu Yanni,graduated philosophy department peking univers...,lecturer,tsinghua university,C_375629
1,T_1,Liu Yanni,graduated philosophy department peking univers...,lecturer,tsinghua university,C_425434
2,T_1,Liu Yanni,graduated philosophy department peking univers...,lecturer,tsinghua university,C_445639
3,T_1,Liu Yanni,graduated philosophy department peking univers...,lecturer,tsinghua university,C_445639
4,T_1,Liu Yanni,graduated philosophy department peking univers...,lecturer,tsinghua university,C_676649


In [25]:
print(f'course_school: {course_school.shape}')
print(course_school.columns.tolist())
print(f'\n')
print(f'course_teacher: {course_teacher.shape}')
print(course_teacher.columns.tolist())

course_school: (3816, 5)
['course_id', 'school_id', 'school_name', 'school_about', 'school_motto']


course_teacher: (97173, 6)
['teacher_id', 'teacher_name', 'teacher_about', 'teacher_job_title', 'teacher_org_name', 'course_id']


In [26]:
course.describe()

,course_id,course_name,course_prerequisites,course_about,course_field
count,3769,3769,3769,3682,547
unique,3769,3123,747,3182,160
top,C_2329163,linear algebra,not stated,full-cycle training plan for improving teachin...,[computer science and technology]
freq,1,9,2570,68,55


In [27]:
print(f'user_course: {user_course.shape}') # Chứa thông tin: mã id học viên - mã id khóa học - thời gian đăng ký 
print(f'user: {user.shape}') # Chứa thông tin: mã id học viên - giới tính 
print(f'course: {course.shape}') # Chứa thông tin: mã id khóa học - tên khóa học - khóa học tiên quyết - mô tả khóa học - lĩnh vực của khóa học

user_course: (11807090, 3)
user: (3330294, 2)
course: (3769, 5)


In [28]:
# Xử lý tập dữ liệu course_teacher 
# Xóa thông tin trùng lặp - gộp thông tin về giảng viên theo từng khóa học 
cb_course_teacher = course_teacher.copy()
print(cb_course_teacher.shape)
cb_course_teacher = cb_course_teacher.drop_duplicates(subset = ['course_id', 'teacher_id'])
cb_course_teacher = cb_course_teacher.groupby(['course_id'])[[
    'teacher_id', 'teacher_name', 'teacher_about', 'teacher_job_title', 'teacher_org_name'
]].agg(list).reset_index()
cb_course_teacher

(97173, 6)


,course_id,teacher_id,teacher_name,teacher_about,teacher_job_title,teacher_org_name
0,C_1001635,"[T_8703, T_8704]","[Zhang Junwei, Yang Chao]","[nan, nan]","[nan, nan]",[xian university electronic science technology...
1,C_1007041,[T_9105],[Peng Dan],[nan],[associate professor],[shenzhen information vocational technical col...
2,C_1007053,[T_9411],[Nie Lei],[teacher garden engineering technology],[professor],[shenzhen information vocational technical col...
3,C_1007072,[T_9411],[Nie Lei],[teacher garden engineering technology],[professor],[shenzhen information vocational technical col...
4,C_1007088,[T_9411],[Nie Lei],[teacher garden engineering technology],[professor],[shenzhen information vocational technical col...
...,...,...,...,...,...,...
13859,C_999396,"[T_7592, T_7593, T_7594]","[Zhang Yaling, Lu Xiaofeng, Wang Zhanmin]",[course leader lecturer professor zhang yaling...,"[professor, associate professor, associate pro...","[xian university technology, xian university t..."
13860,C_999397,"[T_4524, T_4525, T_4526]","[Li Shugang, Cheng Lianhua, Liu Jikun]",[li shugang phd engineering professor doctoral...,"[professor, professor, associate professor]","[xian university science technology, xian univ..."
13861,C_999399,"[T_204, T_871, T_7596, T_7597, T_7598, T_7599,...","[Wang Yuan, Wang Hong, first teacher's name, S...",[wang yuan librarian hold bachelor art master ...,"[librarian, professor, x, lecturer, lecturer, ...","[tsinghua university, kunming medical universi..."
13862,C_999403,"[T_204, T_871, T_7596, T_7597, T_7598, T_7599,...","[Wang Yuan, Wang Hong, first teacher's name, S...",[wang yuan librarian hold bachelor art master ...,"[librarian, professor, x, lecturer, lecturer, ...","[tsinghua university, kunming medical universi..."


In [29]:
# Xử lý tập dữ liệu course_school
# Xóa thông tin mã id trùng lặp - gộp thông tin về trường học theo từng khóa học
cb_course_school = course_school.copy()
print(cb_course_school.shape)
cb_course_school = cb_course_school.drop_duplicates(subset = ['course_id', 'school_id'])
cb_course_school = cb_course_school.groupby(['course_id'])[[
    'school_id', 'school_name', 'school_about', 'school_motto'
]].agg(list).reset_index()
cb_course_school

(3816, 5)


,course_id,school_id,school_name,school_about,school_motto
0,C_1017355,[S_33],[Guangzhou University of Chinese Medicine],[guangzhou university chinese medicine medicin...,[nan]
1,C_1017419,[S_52],[South-central University For Nationalities],[southcentral minzu university referred southc...,[nan]
2,C_1025064,[S_13],[Central South University],[central south university university located c...,[unity knowledge action practical application ...
3,C_1025076,[S_72],[Northeastern University],[northeastern university neu located shenyang ...,[nan]
4,C_1025079,[S_97],[Beijing Sport University],[beijing sport university referred beijing spo...,[nan]
...,...,...,...,...,...
3785,C_955163,[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,[constantly strive selfimprovement virtuous]
3786,C_956128,[S_144],[Shandong University],[shandong university university directly minis...,[nan]
3787,C_956129,[S_492],[Guangzhou Vocational College of Technology & ...,[guangzhou vocational college science technolo...,[nan]
3788,C_956130,[S_11],[South China University of Technology],[south china university technology technology ...,[knowledgeable think carefully discern act dil...


In [30]:
# Kiểm tra tập video 
print(video.head())
video = video.rename(columns = {'ccid': 'video_ccid'})
print(video.isnull().sum())

                               ccid  video_duration   video_id
0  0001603F826A3DDA9C33DC5901307461         509.095    V_93127
1  0001603F826A3DDA9C33DC5901307461         509.095    V_93381
2  0001603F826A3DDA9C33DC5901307461         509.095    V_93474
3  0001603F826A3DDA9C33DC5901307461         509.095   V_645899
4  0001603F826A3DDA9C33DC5901307461         509.095  V_1250778
video_ccid        0
video_duration    0
video_id          0
dtype: int64


In [31]:
# Kiểm tra các cột thông tin trong tập course_video 
print(course_video.shape)
course_video.head(2)

(85787, 2)


,course_id,video_id
0,C_677105,V_93474
1,C_696877,V_472954


In [32]:
# Gộp dữ liệu tập course_video và video 
# Thông tin: lưu các thông tin video giảng dạy của khóa học đó, bao gồm: mã khóa học - mã video thuộc về - thời lượng video - mã video nhỏ bên trong video (ccid)
cb_course_video = pd.merge(course_video, video, on='video_id', how = "left")
cb_course_video.isnull().mean()

course_id         0.0
video_id          0.0
video_ccid        0.0
video_duration    0.0
dtype: float64

In [33]:
# Xử lý tập sau khi gộp cb_course_video 
# Loại bỏ trùng lặp - Gộp các thông tin về video cho từng khóa học 
cols_name = video.columns
cb_course_video = cb_course_video.drop_duplicates()
cb_course_video = cb_course_video.dropna()
cb_course_video_list = cb_course_video.groupby(['course_id'])[cols_name].agg(list).reset_index()
cb_course_video_list

,course_id,video_ccid,video_duration,video_id
0,C_1025064,"[0B63D5E3D4D034E89C33DC5901307461, 0B6DEBC0F80...","[466.575, 402.325, 476.65, 1025.9, 755.075, 42...","[V_1622637, V_1622640, V_1622622, V_1622571, V..."
1,C_1025079,"[0805D199A89E1D799C33DC5901307461, 1026CDC4B72...","[671.97, 925.17, 748.11, 644.25, 543.93, 546.3...","[V_1623000, V_1622990, V_1622999, V_1623003, V..."
2,C_1073350,"[85AED0EB3E502F579C33DC5901307461, C89B9BCF178...","[537.42, 353.135]","[V_1666680, V_4903099]"
3,C_1123814,"[037ADF4715C21A839C33DC5901307461, 09ABCD91037...","[479.741, 629.348, 563.919, 682.89, 634.135, 8...","[V_1685285, V_1685292, V_1685305, V_1685268, V..."
4,C_1169394,"[0086579E5E48FC179C33DC5901307461, 06952EAE1BB...","[649.72, 304.4, 193.64, 493.92, 372.2, 468.04,...","[V_1724743, V_1724514, V_1724866, V_1724794, V..."
...,...,...,...,...
1517,C_949540,"[00CC2D694D4C9AD99C33DC5901307461, 00F9DB5674F...","[702.625, 754.8, 727.15, 673.1, 379.586, 977.3...","[V_2316648, V_2316654, V_3183769, V_3462935, V..."
1518,C_949541,"[0A19E8C9397E5A879C33DC5901307461, 10022BA057E...","[465.175, 398.82, 560.31, 484.5, 632.925, 827....","[V_1505096, V_1505098, V_1505056, V_1505102, V..."
1519,C_949542,"[009BB91833DD44A49C33DC5901307461, 012B7F50301...","[784.77, 713.7, 578.07, 689.01, 624.45, 658.17...","[V_1505228, V_1505190, V_1505250, V_1505241, V..."
1520,C_955163,"[0314DA139E06D2CA9C33DC5901307461, 0A4FA12E2B7...","[515.893, 134.584, 2084.548, 1546.264, 38.356,...","[V_1542395, V_1542782, V_1542410, V_1542374, V..."


In [34]:
# Chứa thông tin: người học xem video nào - tổng thời gian xem video thực tế 
user_video

,user_id,video_id,total_watch_time,actual_watch_time
0,U_2101588,V_6239932,90.000,90.000
1,U_16358526,V_1395637,64.900,64.900
2,U_21473034,V_1358622,37.500,37.500
3,U_16013449,V_7034406,712.000,712.000
4,U_32853687,V_6332628,75.600,75.600
...,...,...,...,...
1980996,U_12422113,V_1358600,558.000,558.000
1980997,U_12979141,V_895909,26.016,26.016
1980998,U_33811817,V_7294536,109.000,109.000
1980999,U_16267593,V_1395633,60.000,60.000


In [35]:
# Gộp thông tin: người học đăng ký khóa học và đã xem video nào trong khóa học đó 
cb_user_course_video = pd.merge(user_video,cb_course_video,on=['video_id'],how='right')
cb_user_course_video

,user_id,video_id,total_watch_time,actual_watch_time,course_id,video_ccid,video_duration
0,NaN,V_93474,NaN,NaN,C_677105,0001603F826A3DDA9C33DC5901307461,509.095
1,NaN,V_472954,NaN,NaN,C_696877,0003DB14A14A53279C33DC5901307461,367.336
2,NaN,V_448757,NaN,NaN,C_696659,0004A5C6F07E369D9C33DC5901307461,492.315
3,NaN,V_5080653,NaN,NaN,C_1784725,0004A5C6F07E369D9C33DC5901307461,492.315
4,NaN,V_555489,NaN,NaN,C_697721,00059EBD1371A6959C33DC5901307461,630.757
...,...,...,...,...,...,...,...
121752,NaN,V_5987059,NaN,NaN,C_1886688,FFF99B018E97FA6E9C33DC5901307461,784.400
121753,NaN,V_140144,NaN,NaN,C_680915,FFFB9B63E30B25919C33DC5901307461,542.000
121754,NaN,V_220033,NaN,NaN,C_681746,FFFE61FFCECC05729C33DC5901307461,554.400
121755,NaN,V_275285,NaN,NaN,C_682190,FFFE61FFCECC05729C33DC5901307461,554.400


In [36]:
# Gom thông tin về người học đăng kí khóa học và các video đã xem 
cols_name = ['video_id','total_watch_time','actual_watch_time','video_ccid','video_duration']
cb_user_course_video = cb_user_course_video.groupby(['user_id','course_id'])[cols_name].agg(list).reset_index()
cb_user_course_video

,user_id,course_id,video_id,total_watch_time,actual_watch_time,video_ccid,video_duration
0,U_1000454,C_2199449,[V_7396061],[450.081],[450.081],[05F564395C35232E9C33DC5901307461],[462.142]
1,U_1002784,C_1756056,"[V_4411676, V_4411678, V_4411680]","[350.20000000000005, 285.40000000000003, 550.0]","[350.20000000000005, 285.40000000000003, 550.0]","[439EAFC2EA7521699C33DC5901307461, 7BACA405FA1...","[379.35, 284.32, 896.41]"
2,U_100281,C_1764866,[V_4677748],[416.1],[452.1],[15E18AD98720C9B89C33DC5901307461],[444.819]
3,U_100281,C_747033,"[V_731722, V_731719, V_731716]","[150.0, 486.0, 62.0]","[150.0, 471.0, 62.0]","[259709F4A24A3B4F9C33DC5901307461, 4CE60DF8B54...","[1040.397, 472.05, 68.57]"
4,U_10033817,C_676642,"[V_1642603, V_1642632, V_1642631]","[10.0, 10.0, 5.0]","[10.0, 10.0, 5.0]","[5846BB9A01701AE89C33DC5901307461, 9BF9C86E729...","[343.2, 370.0, 823.64]"
...,...,...,...,...,...,...,...
11926,U_9968019,C_769273,"[V_768979, V_768978, V_768976, V_768980]","[120.0, 575.0, 101.0, 149.0]","[120.0, 559.0, 86.0, 149.0]","[08D27E716C0B70F89C33DC5901307461, 15E1F223F1C...","[650.675, 643.7, 356.65, 807.6]"
11927,U_996997,C_1774987,[V_4886058],[38.30000000000001],[38.30000000000001],[452016DB4F8D62199C33DC5901307461],[254.001]
11928,U_996997,C_735164,"[V_6293309, V_6293303, V_6293306]","[162.79999999999995, 482.6, 380.0]","[162.79999999999995, 482.6, 380.0]","[9A0D280494C185A49C33DC5901307461, B5FD17CF9EA...","[693.575, 608.955, 799.13]"
11929,U_997041,C_735164,"[V_6293456, V_6293303, V_6293306, V_6293280]","[185.221684, 429.369029, 354.74286299999994, 4...","[185.221684, 429.369029, 354.74286299999994, 4...","[6F5009A0ECE24F0D9C33DC5901307461, B5FD17CF9EA...","[255.616, 608.955, 799.13, 857.747]"


In [37]:
cb_course_video_list.head()

,course_id,video_ccid,video_duration,video_id
0,C_1025064,"[0B63D5E3D4D034E89C33DC5901307461, 0B6DEBC0F80...","[466.575, 402.325, 476.65, 1025.9, 755.075, 42...","[V_1622637, V_1622640, V_1622622, V_1622571, V..."
1,C_1025079,"[0805D199A89E1D799C33DC5901307461, 1026CDC4B72...","[671.97, 925.17, 748.11, 644.25, 543.93, 546.3...","[V_1623000, V_1622990, V_1622999, V_1623003, V..."
2,C_1073350,"[85AED0EB3E502F579C33DC5901307461, C89B9BCF178...","[537.42, 353.135]","[V_1666680, V_4903099]"
3,C_1123814,"[037ADF4715C21A839C33DC5901307461, 09ABCD91037...","[479.741, 629.348, 563.919, 682.89, 634.135, 8...","[V_1685285, V_1685292, V_1685305, V_1685268, V..."
4,C_1169394,"[0086579E5E48FC179C33DC5901307461, 06952EAE1BB...","[649.72, 304.4, 193.64, 493.92, 372.2, 468.04,...","[V_1724743, V_1724514, V_1724866, V_1724794, V..."


In [38]:
print(f'course: {course.shape}')
print(course.columns.tolist())
print(f'course_id.unique: {course['course_id'].nunique()}')
print('\n')

print(f'cb_course_school: {cb_course_school.shape}')
print(cb_course_school.columns.tolist())
print(f'course_id.unique: {cb_course_school['course_id'].nunique()}')
print('\n')

print(f'cb_course_teacher: {cb_course_teacher.shape}')
print(cb_course_teacher.columns.tolist())
print(f'course_id.unique: {cb_course_teacher['course_id'].nunique()}')
print('\n')

print(f'cb_course_video_list: {cb_course_video_list.shape}')
print(cb_course_video_list.columns.tolist())
print(f'course_id.unique: {cb_course_video_list['course_id'].nunique()}')
print('\n')

print(f'cb_user_course_video: {cb_user_course_video.shape}')
print(cb_course_teacher.columns.tolist())
print(f'course_id.unique: {cb_user_course_video['course_id'].nunique()}')
print(f'user_id.unique: {cb_user_course_video['user_id'].nunique()}')

course: (3769, 5)
['course_id', 'course_name', 'course_prerequisites', 'course_about', 'course_field']
course_id.unique: 3769


cb_course_school: (3790, 5)
['course_id', 'school_id', 'school_name', 'school_about', 'school_motto']
course_id.unique: 3790


cb_course_teacher: (13864, 6)
['course_id', 'teacher_id', 'teacher_name', 'teacher_about', 'teacher_job_title', 'teacher_org_name']
course_id.unique: 13864


cb_course_video_list: (1522, 4)
['course_id', 'video_ccid', 'video_duration', 'video_id']
course_id.unique: 1522


cb_user_course_video: (11931, 7)
['course_id', 'teacher_id', 'teacher_name', 'teacher_about', 'teacher_job_title', 'teacher_org_name']
course_id.unique: 458
user_id.unique: 11520


In [39]:
# Gộp các thông tin liên quan đến mỗi khóa học 
# Thông tin: mã id khóa - tên khóa - khóa học tiên quyết - lĩnh vực - mã trường - ... - mã giảng viên - ... - video - thời lượng video  
cb_course = pd.merge(course,cb_course_school,on='course_id',how='left')
cb_course = pd.merge(cb_course,cb_course_teacher,on='course_id',how='left')
cb_course = pd.merge(cb_course,cb_course_video_list,on='course_id',how='left')
cb_course

,course_id,course_name,course_prerequisites,course_about,course_field,school_id,school_name,school_about,school_motto,teacher_id,teacher_name,teacher_about,teacher_job_title,teacher_org_name,video_ccid,video_duration,video_id
0,C_584313,"introduction to ""zi zhi tong jian""",not stated,"through the teacher's guidance, students can g...","[chinese language and literature, history]",[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,[constantly strive selfimprovement virtuous],[T_20],[Zhang Guogang],[zhang guogang male born anqing anhui province...,[professor],[tsinghua university],"[0192C9E042651D369C33DC5901307461, 02B1CA99EAE...","[524.371, 397.52, 677.391, 997.872, 821.917, 6...","[V_867, V_931, V_952, V_901, V_857, V_915, V_8..."
1,C_584329,calculus - limit theory and functions of one v...,not stated,this course is a basic mathematics course in s...,"[applied economics, math, theoretical economic...",[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,[constantly strive selfimprovement virtuous],"[T_21, T_22]","[Hu Zhiming, Zhang Jimin]",[graduated department mathematics peking unive...,"[associate professor, associate professor]","[tsinghua university, tsinghua university]","[01025FA43C787FF09C33DC5901307461, 01ABB469C7E...","[585.519, 446.6, 1216.519, 789.1, 1314.28, 543...","[V_1525, V_1473, V_1470, V_1497, V_1449, V_146..."
2,C_584381,photojournalism,not stated,"master basic photography skills, understand ho...","[art, journalism and communication]",[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,[constantly strive selfimprovement virtuous],[T_23],[Liang Junjian],[associate professor school journalism communi...,[associate professor],[tsinghua university],"[01D0B67D2154D3459C33DC5901307461, 01F0AEBFB50...","[444.56, 204.613, 505.287, 606.654, 700.328, 2...","[V_1822, V_1806, V_1896, V_1869, V_1901, V_186..."
3,C_597208,data mining: theory and algorithms,not stated,the most interesting theory + the most useful ...,[computer science and technology],[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,[constantly strive selfimprovement virtuous],[T_28],[Yuan Bo],[phd computer science university queensland au...,[associate researcher],[tsinghua university],"[03D205A6AEB0B16F9C33DC5901307461, 03F5AF209FA...","[1003.9, 773.66, 886.35, 505.57, 889.61, 930.6...","[V_3025, V_2985, V_2964, V_3074, V_3082, V_306..."
4,C_597225,university computer,not stated,university computer courses will be guided by ...,NaN,[S_6],[Beijing Institute of Technology],[beijing institute technology technology first...,[nan],"[T_41, T_42]","[Li Fengxia, Chen Yufeng]",[li fengxia female professor school computer s...,"[professor, associate professor]","[beijing institute technology, beijing institu...",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3764,C_2338076,(wind project) software engineering,not stated,project blast 2021 [first phase recruitment] i...,NaN,[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,[constantly strive selfimprovement virtuous],"[T_172, T_835]","[Liu Qiang, Liu Lin]",[liu qiang associate professor graduate tutor ...,"[associate professor, associate professor]","[tsinghua university, tsinghua university]","[0213690C975015059C33DC5901307461, 0482FE3877C...","[1108.66, 391.7, 777.06, 729.02, 723.62, 382.2...","[V_8729262, V_8729306, V_8729264, V_8729305, V..."
3765,C_2341259,(wind project) object-oriented programming (c++),not stated,project blast 2021 [first phase recruitment] i...,NaN,[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,[constantly strive selfimprovement virtuous],"[T_38, T_3160]","[Xu Mingxing, Huang Zhenchun]",[xu mingxing phd associate professor person ch...,"[associate professor, associate researcher]","[tsinghua university, tsinghua university]","[05E8B15BD7931AEC9C33DC5901307461, 13E87E6923F...","[439.653, 47

In [40]:
# Tạo cột 'course_text' chứa thông tin: tên khóa học + mô tả khóa học + tên trường + tên giảng viên 
# Ý nghĩa: giúp phân biệt đối với các khóa học có tên giống nhau 
def _join(v):
    # Bỏ giá trị thiếu để không sinh chuỗi "nan" trong văn bản
    if isinstance(v, list):
        return ' '.join(str(x).strip() for x in v if pd.notna(x) and str(x).strip())
    return ''
    
def build_course_text(r):
    name = '' if pd.isna(r['course_name']) else str(r['course_name'])
    school, teacher = _join(r['school_name']), _join(r['teacher_name'])
    about = r['course_about']
    if isinstance(about, str) and about.strip():
        return ' '.join(x for x in [name, school, teacher, about] if x)
    return ' '.join(x for x in [name, school, teacher] if x)   # văn bản rỗng -> tên khóa + trường + giảng viên 

cb_course['course_text'] = cb_course.apply(build_course_text, axis=1)
print((cb_course['course_text'].str.len() == 0).sum(), 'khóa có course_text rỗng')
print(cb_course['course_text'].str.contains(r'\bnan\b', case=False, regex=True).sum(), 'khóa có chữ "nan" trong course_text')
print(cb_course.shape)
cb_course

0 khóa có course_text rỗng
26 khóa có chữ "nan" trong course_text
(3769, 18)


,course_id,course_name,course_prerequisites,course_about,course_field,school_id,school_name,school_about,school_motto,teacher_id,teacher_name,teacher_about,teacher_job_title,teacher_org_name,video_ccid,video_duration,video_id,course_text
0,C_584313,"introduction to ""zi zhi tong jian""",not stated,"through the teacher's guidance, students can g...","[chinese language and literature, history]",[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,[constantly strive selfimprovement virtuous],[T_20],[Zhang Guogang],[zhang guogang male born anqing anhui province...,[professor],[tsinghua university],"[0192C9E042651D369C33DC5901307461, 02B1CA99EAE...","[524.371, 397.52, 677.391, 997.872, 821.917, 6...","[V_867, V_931, V_952, V_901, V_857, V_915, V_8...","introduction to ""zi zhi tong jian"" Tsinghua Un..."
1,C_584329,calculus - limit theory and functions of one v...,not stated,this course is a basic mathematics course in s...,"[applied economics, math, theoretical economic...",[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,[constantly strive selfimprovement virtuous],"[T_21, T_22]","[Hu Zhiming, Zhang Jimin]",[graduated department mathematics peking unive...,"[associate professor, associate professor]","[tsinghua university, tsinghua university]","[01025FA43C787FF09C33DC5901307461, 01ABB469C7E...","[585.519, 446.6, 1216.519, 789.1, 1314.28, 543...","[V_1525, V_1473, V_1470, V_1497, V_1449, V_146...",calculus - limit theory and functions of one v...
2,C_584381,photojournalism,not stated,"master basic photography skills, understand ho...","[art, journalism and communication]",[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,[constantly strive selfimprovement virtuous],[T_23],[Liang Junjian],[associate professor school journalism communi...,[associate professor],[tsinghua university],"[01D0B67D2154D3459C33DC5901307461, 01F0AEBFB50...","[444.56, 204.613, 505.287, 606.654, 700.328, 2...","[V_1822, V_1806, V_1896, V_1869, V_1901, V_186...",photojournalism Tsinghua University Liang Junj...
3,C_597208,data mining: theory and algorithms,not stated,the most interesting theory + the most useful ...,[computer science and technology],[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,[constantly strive selfimprovement virtuous],[T_28],[Yuan Bo],[phd computer science university queensland au...,[associate researcher],[tsinghua university],"[03D205A6AEB0B16F9C33DC5901307461, 03F5AF209FA...","[1003.9, 773.66, 886.35, 505.57, 889.61, 930.6...","[V_3025, V_2985, V_2964, V_3074, V_3082, V_306...",data mining: theory and algorithms Tsinghua Un...
4,C_597225,university computer,not stated,university computer courses will be guided by ...,NaN,[S_6],[Beijing Institute of Technology],[beijing institute technology technology first...,[nan],"[T_41, T_42]","[Li Fengxia, Chen Yufeng]",[li fengxia female professor school computer s...,"[professor, associate professor]","[beijing institute technology, beijing institu...",NaN,NaN,NaN,university computer Beijing Institute of Techn...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3764,C_2338076,(wind project) software engineering,not stated,project blast 2021 [first phase recruitment] i...,NaN,[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,[constantly strive selfimprovement virtuous],"[T_172, T_835]","[Liu Qiang, Liu Lin]",[liu qiang associate professor graduate tutor ...,"[associate professor, associate professor]","[tsinghua university, tsinghua university]","[0213690C975015059C33DC5901307461, 0482FE3877C...","[1108.66, 391.7, 777.06, 729.02, 723.62, 382.2...","[V_8729262, V_8729306, V_8729264, V_8729305, V...",(wind project) software engineering Tsinghua U...
3765,C_2341259,(wind project) object-oriented programming (c++),not stated,project blast 2021 [first phase recruitment] i...,NaN,[S_1],[Tsinghua University],[tsinghua short directly minis

In [41]:
# Gộp dữ liệu học viên và thông tin khóa học đăng ký kèm theo thời gian đăng ký 
combine_data = pd.merge(user_course, cb_course, on = 'course_id', how = 'inner')
combine_data

,user_id,course_id,enroll_time,course_name,course_prerequisites,course_about,course_field,school_id,school_name,school_about,school_motto,teacher_id,teacher_name,teacher_about,teacher_job_title,teacher_org_name,video_ccid,video_duration,video_id,course_text
0,U_22,C_682129,1570876082,software theoretical basis,not stated,"this course introduces formal languages, autom...",NaN,[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,[constantly strive selfimprovement virtuous],[T_2648],[Luo Guiming],[luo guiming received master degree university...,[professor],[tsinghua university],"[00784179599D8C659C33DC5901307461, 036F26D5314...","[1236.79, 1212.67, 1584.55, 1574.35, 376.79, 5...","[V_265396, V_265411, V_265404, V_265442, V_265...",software theoretical basis Tsinghua University...
1,U_24,C_597314,1560360127,chinese language in college - appreciation of ...,not stated,"at present, the talent gap in data science and...",[chinese language and literature],"[S_1, S_14]","[Tsinghua University, Southeast University]",[tsinghua short directly ministry education pe...,"[constantly strive selfimprovement virtuous, s...",[T_113],[Wang Bugao],[wang bugao born yangzhong jiangsu province st...,[distinguished professor],[tsinghua university],"[059F9E3D0C61EC3A9C33DC5901307461, 05BA5BEC636...","[678.88, 1861.28, 280.63, 1488.369, 1192.15, 6...","[V_10741, V_10729, V_10669, V_10636, V_10583, ...",chinese language in college - appreciation of ...
2,U_24,C_597208,1560784961,data mining: theory and algorithms,not stated,the most interesting theory + the most useful ...,[computer science and technology],[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,[constantly strive selfimprovement virtuous],[T_28],[Yuan Bo],[phd computer science university queensland au...,[associate researcher],[tsinghua university],"[03D205A6AEB0B16F9C33DC5901307461, 03F5AF209FA...","[1003.9, 773.66, 886.35, 505.57, 889.61, 930.6...","[V_3025, V_2985, V_2964, V_3074, V_3082, V_306...",data mining: theory and algorithms Tsinghua Un...
3,U_24,C_682129,1570379031,software theoretical basis,not stated,"this course introduces formal languages, autom...",NaN,[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,[constantly strive selfimprovement virtuous],[T_2648],[Luo Guiming],[luo guiming received master degree university...,[professor],[tsinghua university],"[00784179599D8C659C33DC5901307461, 036F26D5314...","[1236.79, 1212.67, 1584.55, 1574.35, 376.79, 5...","[V_265396, V_265411, V_265404, V_265442, V_265...",software theoretical basis Tsinghua University...
4,U_24,C_674910,1570557064,pattern aesthetics and creation,not stated,[national quality course] this course provides...,NaN,[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,[constantly strive selfimprovement virtuous],[T_307],[Nie Yuehua],[nie yuehua member chinese artist association ...,[associate professor],[tsinghua university],NaN,NaN,NaN,pattern aesthetics and creation Tsinghua Unive...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11522935,U_34712115,C_770738,1602487061,data structure (part 2),not stated,project blast 2021 [first phase recruitment] i...,NaN,[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,[constantly strive selfimprovement virtuous],[T_805],[Tang Junhui],[deng junhui professor computer science tsingh...,[professor],[tsinghua university],"[00CE59849A6B87B29C33DC5901307461, 00DDC85FD9E...","[173.09, 312.0, 333.86, 446.9, 93.46, 142.56, ...","[V_784218, V_784314, V_784095, V_784020, V_784...",data structure (part 2) Tsinghua University Ta...
11522936,U_34712115,C_676937,1603225930,c++ language programming advanced,not stated,project blast 2021 [first phase recruitment] i...,[computer science and technology],[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,[constantly strive selfimprovement virtuous],"[T_3

In [42]:
print(f'user_video: {user_video.shape}')
print(user_video.columns.tolist())
print('\n')

print(f'course_video: {course_video.shape}')
print(course_video.columns.tolist())

user_video: (1981001, 4)
['user_id', 'video_id', 'total_watch_time', 'actual_watch_time']


course_video: (85787, 2)
['course_id', 'video_id']


In [43]:
# Gộp dữ liệu hai tập user_video và course_video 
# Ý nghĩa: học viên đăng ký khóa học và xem video nào trong khóa đó 
# Gộp dữ liệu về các video theo course và user 
user_course_video = pd.merge(user_video,course_video,on='video_id',how='inner')
user_course_video = user_course_video.groupby(['user_id','course_id']).agg(list).reset_index()
user_course_video = user_course_video.rename(columns = {'video_id':'watch_id'})
user_course_video

,user_id,course_id,watch_id,total_watch_time,actual_watch_time
0,U_1000454,C_2199449,[V_7396061],[450.081],[450.081]
1,U_1002784,C_1756056,"[V_4411676, V_4411678, V_4411680]","[350.20000000000005, 285.40000000000003, 550.0]","[350.20000000000005, 285.40000000000003, 550.0]"
2,U_100281,C_1764866,[V_4677748],[416.1],[452.1]
3,U_100281,C_747033,"[V_731719, V_731716, V_731722]","[486.0, 62.0, 150.0]","[471.0, 62.0, 150.0]"
4,U_10033817,C_676642,"[V_1642632, V_1642631, V_1642603]","[10.0, 5.0, 10.0]","[10.0, 5.0, 10.0]"
...,...,...,...,...,...
11926,U_9968019,C_769273,"[V_768980, V_768976, V_768978, V_768979]","[149.0, 101.0, 575.0, 120.0]","[149.0, 86.0, 559.0, 120.0]"
11927,U_996997,C_1774987,[V_4886058],[38.30000000000001],[38.30000000000001]
11928,U_996997,C_735164,"[V_6293306, V_6293303, V_6293309]","[380.0, 482.6, 162.79999999999995]","[380.0, 482.6, 162.79999999999995]"
11929,U_997041,C_735164,"[V_6293306, V_6293280, V_6293303, V_6293456]","[354.74286299999994, 499.5096130000002, 429.36...","[354.74286299999994, 499.5096130000002, 429.36..."


In [44]:
# Gộp dữ , cho biết rằng: video thực tế mà học viên xem trong khóa đó 
combine_data = pd.merge(combine_data, user_course_video, on = ['user_id', 'course_id'], how = 'left')
combine_data

,user_id,course_id,enroll_time,course_name,course_prerequisites,course_about,course_field,school_id,school_name,school_about,...,teacher_about,teacher_job_title,teacher_org_name,video_ccid,video_duration,video_id,course_text,watch_id,total_watch_time,actual_watch_time
0,U_22,C_682129,1570876082,software theoretical basis,not stated,"this course introduces formal languages, autom...",NaN,[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,...,[luo guiming received master degree university...,[professor],[tsinghua university],"[00784179599D8C659C33DC5901307461, 036F26D5314...","[1236.79, 1212.67, 1584.55, 1574.35, 376.79, 5...","[V_265396, V_265411, V_265404, V_265442, V_265...",software theoretical basis Tsinghua University...,NaN,NaN,NaN
1,U_24,C_597314,1560360127,chinese language in college - appreciation of ...,not stated,"at present, the talent gap in data science and...",[chinese language and literature],"[S_1, S_14]","[Tsinghua University, Southeast University]",[tsinghua short directly ministry education pe...,...,[wang bugao born yangzhong jiangsu province st...,[distinguished professor],[tsinghua university],"[059F9E3D0C61EC3A9C33DC5901307461, 05BA5BEC636...","[678.88, 1861.28, 280.63, 1488.369, 1192.15, 6...","[V_10741, V_10729, V_10669, V_10636, V_10583, ...",chinese language in college - appreciation of ...,NaN,NaN,NaN
2,U_24,C_597208,1560784961,data mining: theory and algorithms,not stated,the most interesting theory + the most useful ...,[computer science and technology],[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,...,[phd computer science university queensland au...,[associate researcher],[tsinghua university],"[03D205A6AEB0B16F9C33DC5901307461, 03F5AF209FA...","[1003.9, 773.66, 886.35, 505.57, 889.61, 930.6...","[V_3025, V_2985, V_2964, V_3074, V_3082, V_306...",data mining: theory and algorithms Tsinghua Un...,NaN,NaN,NaN
3,U_24,C_682129,1570379031,software theoretical basis,not stated,"this course introduces formal languages, autom...",NaN,[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,...,[luo guiming received master degree university...,[professor],[tsinghua university],"[00784179599D8C659C33DC5901307461, 036F26D5314...","[1236.79, 1212.67, 1584.55, 1574.35, 376.79, 5...","[V_265396, V_265411, V_265404, V_265442, V_265...",software theoretical basis Tsinghua University...,NaN,NaN,NaN
4,U_24,C_674910,1570557064,pattern aesthetics and creation,not stated,[national quality course] this course provides...,NaN,[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,...,[nie yuehua member chinese artist association ...,[associate professor],[tsinghua university],NaN,NaN,NaN,pattern aesthetics and creation Tsinghua Unive...,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11522935,U_34712115,C_770738,1602487061,data structure (part 2),not stated,project blast 2021 [first phase recruitment] i...,NaN,[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,...,[deng junhui professor computer science tsingh...,[professor],[tsinghua university],"[00CE59849A6B87B29C33DC5901307461, 00DDC85FD9E...","[173.09, 312.0, 333.86, 446.9, 93.46, 142.56, ...","[V_784218, V_784314, V_784095, V_784020, V_784...",data structure (part 2) Tsinghua University Ta...,NaN,NaN,NaN
11522936,U_34712115,C_676937,1603225930,c++ language programming advanced,not stated,project blast 2021 [first phase recruitment] i...,[computer science and technology],[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,...,[xu mingxing phd associate professor person ch...,"[associate professor, professor, associate res...","[tsinghua university, tsinghua university, tsi...","[0086579E5E48FC179C33DC5901307461, 0814FA18F28...","[649.72, 193.64, 493.92, 372.2, 468.04, 230.24...","[V_69237, V_69353, V_69285, V_69228, V_69217, ...",c++ language programming adva

In [45]:
# Kiểm tra tập khái niệm có trong từng khóa học 
print(course_concept.shape)
course_concept.head()

(451073, 4)


,concept_id,course_id,concept_name,concept_field
0,K_n号计数器_控制科学与工程,C_681460,Counter number n,control science and engineering
1,K_串行链接_控制科学与工程,C_681460,Serial link,control science and engineering
2,K_热继电器按钮_控制科学与工程,C_681460,Thermal relay button,control science and engineering
3,K_谐波污染_控制科学与工程,C_681460,Harmonic pollution,control science and engineering
4,K_偶校验转换_控制科学与工程,C_681460,Even parity conversion,control science and engineering


In [46]:
# Kiểm tra các tên khái niệm còn thiếu - chưa thể dịch được 
null_concept = course_concept[course_concept['concept_field'].isnull()| course_concept['concept_name'].isnull()].drop_duplicates().reset_index(drop=True)
null_concept

,concept_id,course_id,concept_name,concept_field
0,K_国际电报电话咨询委员会_计算机科学与技术,C_707083,NaN,computer science and technology
1,K_国际电报电话咨询委员会_计算机科学与技术,C_697379,NaN,computer science and technology
2,K_有理式_数学,C_682271,NaN,mathematics
3,K_有理式_数学,C_782490,NaN,mathematics
4,K_+运算符_计算机科学与技术,C_2341225,NaN,computer science and technology
...,...,...,...,...
59,K_实际成本_情报与档案管理,C_854667,NaN,information and archives management
60,K_脊髓缺血_心理学,C_696586,NaN,psychology
61,K_长春花_园艺学,C_682388,NaN,horticulture
62,K_事前控制_环境科学与工程,C_676890,NaN,environmental science and engineering


In [47]:
concept_id = null_concept['concept_id'].unique()
concept_id.tolist()

['K_国际电报电话咨询委员会_计算机科学与技术',
 'K_有理式_数学',
 'K_+运算符_计算机科学与技术',
 'K_稳定矩阵_数学',
 'K_线性时变系统_数学',
 'K_垂直分量_计算机科学与技术',
 'K_乙肝表面抗原_生物学',
 'K_螺旋形_化学',
 'K_氢氧化锂_化学',
 'K_基因排列_基础医学',
 'K_除磷_水利工程',
 'K_污水处理过程_水利工程',
 'K_阳离子交换器_水利工程',
 'K_自行车_机械工程',
 'K_环境致癌物_临床医学',
 'K_弹簧片_系统科学',
 'K_科技学院_教育学',
 'K_nan _计算机科学与技术',
 'K_意大利蜜蜂_生物学',
 'K_沉默的螺旋_新闻传播学',
 'K_生态群落_生物学',
 'K_文化冲突_教育学',
 'K_概率抽样_社会学',
 'K_实际成本_情报与档案管理',
 'K_脊髓缺血_心理学',
 'K_长春花_园艺学',
 'K_事前控制_环境科学与工程',
 'K_短期供给曲线_理论经济学']

In [48]:
# Dịch tay các trường course_name mà vẫn chưa được dịch 
course_concept.loc[course_concept['concept_id'] == 'K_国际电报电话咨询委员会_计算机科学与技术', 'concept_name'] = 'international telegraph and telephone consultative committee'
course_concept.loc[course_concept['concept_id'] == 'K_有理式_数学', 'concept_name'] = 'rational expression'
course_concept.loc[course_concept['concept_id'] == 'K_+运算符_计算机科学与技术', 'concept_name'] = 'operator'
course_concept.loc[course_concept['concept_id'] == 'K_稳定矩阵_数学', 'concept_name'] = 'stable matrix'
course_concept.loc[course_concept['concept_id'] == 'K_线性时变系统_数学', 'concept_name'] = 'linear time-varying system'
course_concept.loc[course_concept['concept_id'] == 'K_垂直分量_计算机科学与技术', 'concept_name'] = 'vertical component'
course_concept.loc[course_concept['concept_id'] == 'K_乙肝表面抗原_生物学', 'concept_name'] = 'hepatitis b surface antigen'
course_concept.loc[course_concept['concept_id'] == 'K_螺旋形_化学', 'concept_name'] = 'spiral'
course_concept.loc[course_concept['concept_id'] == 'K_氢氧化锂_化学', 'concept_name'] = 'lithium hydroxide'
course_concept.loc[course_concept['concept_id'] == 'K_基因排列_基础医学', 'concept_name'] = 'gene arrangement'
course_concept.loc[course_concept['concept_id'] == 'K_除磷_水利工程', 'concept_name'] = 'phosphorus removal'
course_concept.loc[course_concept['concept_id'] == 'K_污水处理过程_水利工程', 'concept_name'] = 'wastewater treatment process'
course_concept.loc[course_concept['concept_id'] == 'K_阳离子交换器_水利工程', 'concept_name'] = 'cation exchanger'
course_concept.loc[course_concept['concept_id'] == 'K_自行车_机械工程', 'concept_name'] = 'bicycle'
course_concept.loc[course_concept['concept_id'] == 'K_环境致癌物_临床医学', 'concept_name'] = 'environmental carcinogens'
course_concept.loc[course_concept['concept_id'] == 'K_弹簧片_系统科学', 'concept_name'] = 'spring piece'
course_concept.loc[course_concept['concept_id'] == 'K_nan _计算机科学与技术', 'concept_name'] = 'nan value'
course_concept.loc[course_concept['concept_id'] == 'K_意大利蜜蜂_生物学', 'concept_name'] = 'italian honeybee'
course_concept.loc[course_concept['concept_id'] == 'K_沉默的螺旋_新闻传播学', 'concept_name'] = 'the spiral of silence'
course_concept.loc[course_concept['concept_id'] == 'K_生态群落_生物学', 'concept_name'] = 'ecological community'
course_concept.loc[course_concept['concept_id'] == 'K_文化冲突_教育学', 'concept_name'] = 'cultural clash'
course_concept.loc[course_concept['concept_id'] == 'K_概率抽样_社会学', 'concept_name'] = 'probability sampling'
course_concept.loc[course_concept['concept_id'] == 'K_实际成本_情报与档案管理', 'concept_name'] = 'actual cost'
course_concept.loc[course_concept['concept_id'] == 'K_脊髓缺血_心理学', 'concept_name'] = 'spinal cord ischemia'
course_concept.loc[course_concept['concept_id'] == 'K_长春花_园艺学', 'concept_name'] = 'periwinkle'
course_concept.loc[course_concept['concept_id'] == 'K_事前控制_环境科学与工程', 'concept_name'] = 'prevention control'
course_concept.loc[course_concept['concept_id'] == 'K_短期供给曲线_理论经济学', 'concept_name'] = 'short-run supply curve'
course_concept.loc[course_concept['concept_id'] == 'K_科技学院_教育学', 'concept_name'] = 'institute of technology'

In [49]:
# Kiểm tra lại + Xóa trùng lặp 
course_concept[course_concept['concept_field'].isnull()| course_concept['concept_name'].isnull()].drop_duplicates().reset_index(drop=True)

,concept_id,course_id,concept_name,concept_field


In [50]:
print(course_concept.shape)
course_concept.head()

(451073, 4)


,concept_id,course_id,concept_name,concept_field
0,K_n号计数器_控制科学与工程,C_681460,Counter number n,control science and engineering
1,K_串行链接_控制科学与工程,C_681460,Serial link,control science and engineering
2,K_热继电器按钮_控制科学与工程,C_681460,Thermal relay button,control science and engineering
3,K_谐波污染_控制科学与工程,C_681460,Harmonic pollution,control science and engineering
4,K_偶校验转换_控制科学与工程,C_681460,Even parity conversion,control science and engineering


In [51]:
# Gộp các khái niệm có trong một khóa học 
cb_course_concept = course_concept.copy()
cb_course_concept = cb_course_concept.groupby('course_id').agg(list).reset_index()
cb_course_concept

,course_id,concept_id,concept_name,concept_field
0,C_1169394,"[K_数据元素_计算机科学与技术, K_基类的构造函数_计算机科学与技术, K_ostrea...","[Data elements, Base class constructor, ostrea...","[computer science and technology, computer sci..."
1,C_1410156,"[K_程序_计算机科学与技术, K_库函数_计算机科学与技术, K_访问控制_计算机科学与技...","[program, library functions, Access Control, W...","[computer science and technology, computer sci..."
2,C_1429045,"[K_氯丙嗪_临床医学, K_系统综述研究_临床医学, K_共识的方法_临床医学, K_证据...","[Chlorpromazine, Systematic review study, cons...","[clinical medicine, clinical medicine, clinica..."
3,C_1742805,"[K_数据元素_计算机科学与技术, K_程序_计算机科学与技术, K_访问控制_计算机科学与...","[Data elements, program, Access Control, linea...","[computer science and technology, computer sci..."
4,C_1749733,"[K_formula:_计算机科学与技术, K_android _数学, K_hash_计算...","[formula:, Android, hash, mobile phones, ρ, co...","[computer science and technology, mathematics,..."
...,...,...,...,...
882,C_890259,"[K_推荐_管理科学与工程, K_企业的整合能力_管理科学与工程, K_企业战略承诺的形式_...","[recommend, Enterprise integration capabilitie...","[management science and engineering, managemen..."
883,C_901947,"[K_durin_公共卫生与预防医学, K_beats fast_公共卫生与预防医学, K_...","[durin, beats fast, measles, her privacy, sex ...","[public health and preventive medicine, public..."
884,C_922854,"[K_建筑三维模型_建筑学, K_石膏板_建筑学, K_条形基础_建筑学, K_前室_建筑学...","[3D architectural model, plasterboard, strip f...","[architecture, architecture, architecture, arc..."
885,C_928746,"[K_阿姆达尔定律_计算机科学与技术, K_无线收发机_计算机科学与技术, K_程序_计算机...","[Amdahl's Law, wireless transceiver, program, ...","[computer science and technology, computer sci..."


In [52]:
# Thêm các trường liên quan đến khái niệm cho từng khóa học 
combine_data = pd.merge(combine_data, cb_course_concept, on = 'course_id', how = 'left')
combine_data

,user_id,course_id,enroll_time,course_name,course_prerequisites,course_about,course_field,school_id,school_name,school_about,...,video_ccid,video_duration,video_id,course_text,watch_id,total_watch_time,actual_watch_time,concept_id,concept_name,concept_field
0,U_22,C_682129,1570876082,software theoretical basis,not stated,"this course introduces formal languages, autom...",NaN,[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,...,"[00784179599D8C659C33DC5901307461, 036F26D5314...","[1236.79, 1212.67, 1584.55, 1574.35, 376.79, 5...","[V_265396, V_265411, V_265404, V_265442, V_265...",software theoretical basis Tsinghua University...,NaN,NaN,NaN,"[K_归纳假设_计算机科学与技术, K_空语言的问题_计算机科学与技术, K_不可判定问题_...","[Inductive hypothesis, The problem of empty la...","[computer science and technology, computer sci..."
1,U_24,C_597314,1560360127,chinese language in college - appreciation of ...,not stated,"at present, the talent gap in data science and...",[chinese language and literature],"[S_1, S_14]","[Tsinghua University, Southeast University]",[tsinghua short directly ministry education pe...,...,"[059F9E3D0C61EC3A9C33DC5901307461, 05BA5BEC636...","[678.88, 1861.28, 280.63, 1488.369, 1192.15, 6...","[V_10741, V_10729, V_10669, V_10636, V_10583, ...",chinese language in college - appreciation of ...,NaN,NaN,NaN,"[K_索取荆州_中国语言文学, K_你有没有听到我_中国语言文学, K_特立独行的猪_中国语...","[Demand Jingzhou, Did you hear me?, The Maveri...","[chinese language and literature, chinese lang..."
2,U_24,C_597208,1560784961,data mining: theory and algorithms,not stated,the most interesting theory + the most useful ...,[computer science and technology],[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,...,"[03D205A6AEB0B16F9C33DC5901307461, 03F5AF209FA...","[1003.9, 773.66, 886.35, 505.57, 889.61, 930.6...","[V_3025, V_2985, V_2964, V_3074, V_3082, V_306...",data mining: theory and algorithms Tsinghua Un...,NaN,NaN,NaN,"[K_data visualization_计算机科学与技术, K_an address_计...","[data visualization, an address, use data, sea...","[computer science and technology, computer sci..."
3,U_24,C_682129,1570379031,software theoretical basis,not stated,"this course introduces formal languages, autom...",NaN,[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,...,"[00784179599D8C659C33DC5901307461, 036F26D5314...","[1236.79, 1212.67, 1584.55, 1574.35, 376.79, 5...","[V_265396, V_265411, V_265404, V_265442, V_265...",software theoretical basis Tsinghua University...,NaN,NaN,NaN,"[K_归纳假设_计算机科学与技术, K_空语言的问题_计算机科学与技术, K_不可判定问题_...","[Inductive hypothesis, The problem of empty la...","[computer science and technology, computer sci..."
4,U_24,C_674910,1570557064,pattern aesthetics and creation,not stated,[national quality course] this course provides...,NaN,[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,...,NaN,NaN,NaN,pattern aesthetics and creation Tsinghua Unive...,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11522935,U_34712115,C_770738,1602487061,data structure (part 2),not stated,project blast 2021 [first phase recruitment] i...,NaN,[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,...,"[00CE59849A6B87B29C33DC5901307461, 00DDC85FD9E...","[173.09, 312.0, 333.86, 446.9, 93.46, 142.56, ...","[V_784218, V_784314, V_784095, V_784020, V_784...",data structure (part 2) Tsinghua University Ta...,NaN,NaN,NaN,"[K_比较器_计算机科学与技术, K_归纳假设_计算机科学与技术, K_next表构造算法_...","[comparator, Inductive hypothesis, Next list c...","[computer science and technology, computer sci..."
11522936,U_34712115,C_676937,1603225930,c++ language programming advanced,not stated,project blast 2021 [first phase recruitment] i...,[computer science and technology],[S_1],[Tsinghua University],[tsinghua short directly ministry education pe...,...,"[0086579E5E48FC179C33DC5901307461, 0814FA18F28...","[649.72, 193.64, 493.92, 372.2, 468.04, 230.

In [53]:
combine_data.columns.tolist()

['user_id',
 'course_id',
 'enroll_time',
 'course_name',
 'course_prerequisites',
 'course_about',
 'course_field',
 'school_id',
 'school_name',
 'school_about',
 'school_motto',
 'teacher_id',
 'teacher_name',
 'teacher_about',
 'teacher_job_title',
 'teacher_org_name',
 'video_ccid',
 'video_duration',
 'video_id',
 'course_text',
 'watch_id',
 'total_watch_time',
 'actual_watch_time',
 'concept_id',
 'concept_name',
 'concept_field']

In [54]:
# Gộp với tập User -> Thêm cột 'gender'
combine_data = pd.merge(combine_data, user, on = 'user_id', how = 'left')
print(combine_data.shape)

(11522940, 27)


In [55]:
print(combine_data.shape)
print(combine_data['user_id'].nunique())
print(combine_data['course_id'].nunique())

(11522940, 27)
3211039
3769


In [56]:
# Danh sách các cột cần xử lý
list_columns = [
    'school_about',
    'school_motto',
    'teacher_name',
    'teacher_about',
    'teacher_job_title',
    'teacher_org_name'
]

# Hàm xử lý danh sách, thay giá trị khuyết bằng "unknown"
def fill_missing_in_list(lst):
    if isinstance(lst, list):
        return ["unknown" if pd.isna(item) else item for item in lst]
    return lst  # Giữ nguyên nếu không phải list

# Áp dụng cho từng cột trong danh sách
for col in list_columns:
    combine_data[col] = combine_data[col].apply(fill_missing_in_list)

In [57]:
print(combine_data.duplicated(['user_id', 'course_id']).sum(), 'dòng trùng (user, course)')

0 dòng trùng (user, course)


In [58]:
# Thống kê tỷ lệ null đối với từng thuộc tính 
missing_data = combine_data.isnull().sum().reset_index()
missing_data.columns = ['col', 'null']
missing_data['per_null'] = ((missing_data['null'] / combine_data.shape[0]) * 100).round(5)
missing_data

,col,null,per_null
0,user_id,0,0.00000
1,course_id,0,0.00000
2,enroll_time,0,0.00000
3,course_name,0,0.00000
4,course_prerequisites,0,0.00000
5,course_about,8343,0.07240
6,course_field,8450231,73.33398
7,school_id,48122,0.41762
8,school_name,48122,0.41762
9,school_about,48122,0.41762


In [59]:
course_data = combine_data[['course_id', 'course_name', 'course_prerequisites', 'course_about', 'course_field',
                      'course_text', 'school_name', 'school_about', 'school_motto',
                      'teacher_name', 'teacher_about', 'teacher_job_title', 'teacher_org_name',
                      'concept_name', 'concept_id']].drop_duplicates(subset = ['course_id', 'course_name']).reset_index()
print(course_data.shape)

(3769, 16)


In [60]:
scen = "course_data"
output_dir = os.path.join("/kaggle/working", scen)
name = "course_data"

os.makedirs(output_dir, exist_ok=True)
course_data.to_parquet(os.path.join(output_dir, f"{name}.parquet"), index=False, engine="pyarrow")
print(f"Đã lưu: {name}.parquet")

shutil.make_archive("/kaggle/working/course_data", 'zip', "/kaggle/working/course_data")
print("Đã nén xong!")
FileLink('course_data.zip')

Đã lưu: course_data.parquet
Đã nén xong!


/kaggle/working/course_data.zip

In [61]:
#scen = "combine_data"
#output_dir = os.path.join("/kaggle/working", scen)
#name = "combine_data"

#os.makedirs(output_dir, exist_ok=True)
#combine_data.to_parquet(os.path.join(output_dir, f"{name}.parquet"), index=False, engine="pyarrow")
#print(f"Đã lưu: {name}.parquet")

#shutil.make_archive("/kaggle/working/combine_data", 'zip', "/kaggle/working/combine_data")
#print("Đã nén xong!")
#FileLink('combine_data.zip')

In [62]:
# Lấy ra tập dữ liệu tương tác: 'user_id', 'course_id', 'enroll_time'
# Mỗi cặp (học viên, khóa học) chỉ giữ một dòng - lần đăng ký sớm nhất
interactions = (combine_data[['user_id', 'course_id', 'enroll_time']]
                .sort_values('enroll_time', kind='stable')
                .drop_duplicates(subset=['user_id', 'course_id'], keep='first')
                .reset_index(drop=True))
print(interactions.shape)
interactions.head()

(11522940, 3)


,user_id,course_id,enroll_time
0,U_9137692,C_584313,1557326901
1,U_9137692,C_584329,1557327704
2,U_361418,C_584329,1557328370
3,U_69,C_584329,1557328556
4,U_69,C_584313,1557329610


In [63]:
# Kiểm tra bắt buộc: không còn cặp (học viên, khóa học) trùng
n_dup = interactions.duplicated(subset=['user_id', 'course_id']).sum()
assert n_dup == 0, f'Còn {n_dup} cặp (học viên, khóa học) trùng'
print(f"Số học viên: {interactions['user_id'].nunique():,} | Số khóa: {interactions['course_id'].nunique():,} | Số tương tác: {len(interactions):,}")

Số học viên: 3,211,039 | Số khóa: 3,769 | Số tương tác: 11,522,940


In [64]:
scen = "full_interactions"
output_dir = os.path.join("/kaggle/working", scen)
name = "interactions_full"

os.makedirs(output_dir, exist_ok=True)
interactions.to_parquet(os.path.join(output_dir, f"{name}.parquet"), index=False, engine="pyarrow")
print(f"Đã lưu: {name}.parquet")

shutil.make_archive("/kaggle/working/full_interactions", 'zip', "/kaggle/working/full_interactions")
print("Đã nén xong!")
FileLink('full_interactions.zip')

Đã lưu: interactions_full.parquet
Đã nén xong!


/kaggle/working/full_interactions.zip

In [65]:
print('Completed!')

Completed!
